# NB_SecurityDetails_BronzeToSilver  
**Source:** `AI_Medallion.Bronze.security`  
**Target:** `AI_Medallion.Silver.security`  
**Grain:** One row per Bronze row with a non-null Email (no deduplication)  
**Load type:** Overwrite  
**Rule:** Standardize column names (prefix `security_`, snake_case), TRIM all text,  
cast `execution_date` to `timestamp`. Codigo, Conocido and Ticket are kept as string.  
Text "NULL" values are kept as received.

In [1]:
# Workspace folder: C:\\spark-dev\\CoE_transformation_framework (see README)
import os
from pathlib import Path


os.environ["HADOOP_HOME"] = r"C:\hadoop\hadoop-3.5.0"
os.environ["HADOOP_PREFIX"] = os.environ["HADOOP_HOME"]
os.environ["PATH"] = os.path.join(os.environ["HADOOP_HOME"], "bin") + os.pathsep + os.environ["PATH"]
 
print("HADOOP_HOME:", os.environ["HADOOP_HOME"])
print("HADOOP_PREFIX:", os.environ["HADOOP_PREFIX"])
print("winutils exists:", os.path.isfile(
    os.path.join(os.environ["HADOOP_HOME"], "bin", "winutils.exe")
))

print("HADOOP_HOME:", os.environ["HADOOP_HOME"])
print("HADOOP_PREFIX:", os.environ["HADOOP_PREFIX"])
print("winutils exists:", os.path.isfile(
    os.path.join(os.environ["HADOOP_HOME"], "bin", "winutils.exe")
))

HADOOP_HOME: C:\hadoop\hadoop-3.5.0
HADOOP_PREFIX: C:\hadoop\hadoop-3.5.0
winutils exists: True
HADOOP_HOME: C:\hadoop\hadoop-3.5.0
HADOOP_PREFIX: C:\hadoop\hadoop-3.5.0
winutils exists: True


In [2]:
_cwd = Path.cwd()
if not (_cwd / "Common" / "bootstrap.py").is_file():
    _repo = _cwd.parent
    if (_repo / "Common" / "bootstrap.py").is_file():
        os.chdir(_repo)

In [3]:
REBUILD_MEDALLION_CACHE = True  # True: rebuild; False: use cache
os.environ.setdefault("MEDALLION_DEBUG_READ", "1")
from Common.io import sync_rebuild_medallion_cache_env

sync_rebuild_medallion_cache_env(REBUILD_MEDALLION_CACHE)

In [4]:
import sys
print(sys.executable)

c:\spark-dev\CoE_transformation_framework\.venv\Scripts\python.exe


In [5]:
%run ./Common/bootstrap

Bootstrap OK: runtime=local mode=created spark=3.5.4 pyspark=3.5.4 path_reads=True onelake_read=deltalake-scan
Repo root: C:\spark-dev\CoE_Transformation_Framework_copilot
spark.jars.packages=io.delta:delta-spark_2.12:3.2.0,org.apache.hadoop:hadoop-azure:3.3.4,com.azure:azure-storage-blob:12.25.1
OneLake reads: deltalake scan() + az login -> Spark (column mapping; avoids local PySpark abfss hang).
TABLE_CURSOR_BRONZE -> abfss://4cecee03-0629-4081-8ae8-b749ee04d43f@onelake.dfs.fabric.microsoft.com/1ee46463-2495-44f5-9dc9-d635d0067483/Tables/Bronze/cursor_usage
Tip: set MEDALLION_DEBUG_READ=1 before bootstrap to log read progress.
Tip: large tables — read_table() one at a time; transient Azure errors auto-retry.
Local table cache ON (default): parquet under ~/.fabric/medallion_cache/. Set REBUILD_MEDALLION_CACHE = True in settings to force OneLake. reload_io_helpers(globals()) after git pull — no kernel restart.
OneLake writes: deltalake + az login (merge/overwrite). Set MEDALLION_ONELAK

In [6]:
# TODO: move to Common/config.py if not defined there yet
TABLE_SECURITY_BRONZE = globals().get(
    "TABLE_SECURITY_BRONZE", "AI_Medallion.Bronze.security"
)
TABLE_SECURITY_SILVER = globals().get(
    "TABLE_SECURITY_SILVER", "AI_Medallion.Silver.security"
)

In [7]:
bronze = read_table_broken_lineage(TABLE_SECURITY_BRONZE, "security_bronze")
bronze.printSchema()

read_table_broken_lineage: scan -> local cache (AI_Medallion.Bronze.security)
read_table: deltalake scan abfss://4cecee03-0629-4081-8ae8-b749ee04d43f@onelake.dfs.fabric.microsoft.com/1e… (pass 1/6)
read_table: OneLake object-store concurrency limit=1 (raise MEDALLION_OBJECT_STORE_CONCURRENCY if reads are too slow)
read_table: deltalake scan() streaming batches…
read_table: scan streaming… 295 rows from OneLake so far
read_table: scan finished (295 rows)
read_table: opening local parquet cache (295 rows) -> C:\Users\soukelkr\.fabric\medallion_cache\AI_Medallion__Bronze__security
read_table_broken_lineage: opened cache after scan (security_bronze)
root
 |-- Email: string (nullable = true)
 |-- Report: string (nullable = true)
 |-- Access: string (nullable = true)
 |-- Codigo: string (nullable = true)
 |-- Conocido: string (nullable = true)
 |-- Nombre Completo: string (nullable = true)
 |-- Pais: string (nullable = true)
 |-- Job Title: string (nullable = true)
 |-- Ticket: string (nulla

In [8]:
from Common.transforms import _bronze_column


def _txt(*names):
    """TRIM(column) as string; accepts the real bronze name or snake_case variants."""
    return F.trim(_bronze_column(bronze, *names).cast("string"))


# execution_date: string or timestamp in Bronze -> timestamp in Silver
_execution_date = _bronze_column(bronze, "execution_date").cast("timestamp")

In [9]:
silver = (
    bronze.select(
        _txt("Email", "email").alias("security_email"),
        _txt("Report", "report").alias("security_report"),
        _txt("Access", "access").alias("security_access"),
        _txt("Codigo", "codigo").alias("security_codigo"),
        _txt("Conocido", "conocido").alias("security_conocido"),
        _txt("Nombre Completo", "NombreCompleto", "nombre_completo").alias("security_nombre_completo"),
        _txt("Pais", "pais").alias("security_pais"),
        _txt("Job Title", "JobTitle", "job_title").alias("security_job_title"),
        _txt("Ticket", "ticket").alias("security_ticket"),
        _txt("Division", "division").alias("security_division"),
        _txt("AccessScope", "access_scope").alias("security_access_scope"),
        _execution_date.alias("security_execution_date"),
    )
    .filter(F.col("security_email").isNotNull())  # Silver Scope: where Email is not null
    # No dropDuplicates: the file can contain repeated emails with different access rows.
)

In [10]:
silver.printSchema()
silver.show(10, truncate=False)
print("bronze rows:", bronze.count(), "| silver rows:", silver.count())

# Execution date check: rows where the cast failed
bad_dates = silver.filter(
    F.col("security_execution_date").isNull()
).count()
print("security_execution_date null after cast:", bad_dates)

# Text "NULL" values kept as received (info for the data owner)
text_null_email = silver.filter(F.upper(F.col("security_email")) == "NULL").count()
print('security_email = text "NULL":', text_null_email)

root
 |-- security_email: string (nullable = true)
 |-- security_report: string (nullable = true)
 |-- security_access: string (nullable = true)
 |-- security_codigo: string (nullable = true)
 |-- security_conocido: string (nullable = true)
 |-- security_nombre_completo: string (nullable = true)
 |-- security_pais: string (nullable = true)
 |-- security_job_title: string (nullable = true)
 |-- security_ticket: string (nullable = true)
 |-- security_division: string (nullable = true)
 |-- security_access_scope: string (nullable = true)
 |-- security_execution_date: timestamp (nullable = true)

+----------------------------------+---------------+---------------+----------------+-----------------+------------------------------+-------------+------------------+---------------+-----------------+---------------------+--------------------------+
|security_email                    |security_report|security_access|security_codigo |security_conocido|security_nombre_completo      |security_pais|s

In [11]:
apply_local_merge_spark_conf(spark)

write_full_table(silver, TABLE_SECURITY_SILVER)

show_sample(silver.limit(10))

publish_merge_staging: AI_Medallion.Silver.security -> C:\Users\soukelkr\.fabric\merge_staging\AI_Medallion__Silver__security
materialize staging: Spark JVM parquet (8 partitions)
deltalake write mode=overwrite abfss://4cecee03-0629-4081-8ae8-b749ee04d43f@onelake.dfs.fabric.microsoft.com/1ee46463-2495-44f5-9dc9-d635d0067483/Tables/Silver/security (local staging -> OneLake)...
deltalake write chunk 1 (37 rows) mode=overwrite
deltalake write chunk 2 (36 rows) mode=append
deltalake write chunk 3 (37 rows) mode=append
deltalake write chunk 4 (37 rows) mode=append
deltalake write chunk 5 (37 rows) mode=append
deltalake write chunk 6 (37 rows) mode=append
deltalake write chunk 7 (37 rows) mode=append
deltalake write chunk 8 (37 rows) mode=append
deltalake write done


DataFrame[security_email: string, security_report: string, security_access: string, security_codigo: string, security_conocido: string, security_nombre_completo: string, security_pais: string, security_job_title: string, security_ticket: string, security_division: string, security_access_scope: string, security_execution_date: timestamp]